In [4]:
!pip install pandas rapidfuzz ipywidgets openpyxl
import pandas as pd
from rapidfuzz import fuzz
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display, clear_output
def run_matching(vendor_df, employee_df, v_col, e_col, threshold):
    results = []

    for _, v_row in vendor_df.iterrows():
        for _, e_row in employee_df.iterrows():
            v_val = str(v_row[v_col])
            e_val = str(e_row[e_col])

            score = fuzz.token_set_ratio(v_val, e_val)

            if score >= threshold:
                row = {
                    "MATCH_TYPE": "FUZZY" if score < 100 else "EXACT",
                    "MATCH_FIELD": f"{v_col} vs {e_col}",
                    "MATCH_SCORE": score
                }

                for c in vendor_df.columns:
                    row[f"VENDOR_{c}"] = v_row[c]

                for c in employee_df.columns:
                    row[f"EMPLOYEE_{c}"] = e_row[c]

                results.append(row)

    return pd.DataFrame(results)
vendor_uploader = widgets.FileUpload(
    accept='.csv', multiple=False, description="Upload Vendor CSV"
)

employee_uploader = widgets.FileUpload(
    accept='.csv', multiple=False, description="Upload Employee CSV"
)

display(widgets.VBox([
    widgets.HTML("<h3>Upload Data</h3>"),
    vendor_uploader,
    employee_uploader
]))
v_col_dropdown = widgets.Dropdown(description="Vendor Field:")
e_col_dropdown = widgets.Dropdown(description="Employee Field:")

threshold_slider = widgets.IntSlider(
    value=80, min=60, max=100, step=1,
    description="Threshold (%)"
)

display(widgets.VBox([
    v_col_dropdown,
    e_col_dropdown,
    threshold_slider
]))
def load_columns(change=None):
    if vendor_uploader.value and employee_uploader.value:
        vendor_df = pd.read_csv(list(vendor_uploader.value.values())[0]["content"])
        employee_df = pd.read_csv(list(employee_uploader.value.values())[0]["content"])

        vendor_df.columns = vendor_df.columns.str.upper().str.strip()
        employee_df.columns = employee_df.columns.str.upper().str.strip()

        v_col_dropdown.options = vendor_df.columns.tolist()
        e_col_dropdown.options = employee_df.columns.tolist()

vendor_uploader.observe(load_columns, names="value")
employee_uploader.observe(load_columns, names="value")
run_button = widgets.Button(
    description="RUN ANALYSIS",
    button_style="primary"
)

output = widgets.Output()

def on_run_clicked(b):
    with output:
        clear_output()

        if not vendor_uploader.value or not employee_uploader.value:
            print("❌ Please upload both CSV files.")
            return

        vendor_df = pd.read_csv(list(vendor_uploader.value.values())[0]["content"])
        employee_df = pd.read_csv(list(employee_uploader.value.values())[0]["content"])

        vendor_df.columns = vendor_df.columns.str.upper().str.strip()
        employee_df.columns = employee_df.columns.str.upper().str.strip()

        result_df = run_matching(
            vendor_df,
            employee_df,
            v_col_dropdown.value,
            e_col_dropdown.value,
            threshold_slider.value
        )

        if result_df.empty:
            print("✅ No conflicts found.")
        else:
            print(f"⚠️ Conflicts Found: {len(result_df)}")
            display(result_df)

run_button.on_click(on_run_clicked)

display(run_button, output)
export_button = widgets.Button(
    description="Download Excel Report",
    button_style="success"
)

def export_excel(b):
    if not vendor_uploader.value or not employee_uploader.value:
        print("No data to export.")
        return

    vendor_df = pd.read_csv(list(vendor_uploader.value.values())[0]["content"])
    employee_df = pd.read_csv(list(employee_uploader.value.values())[0]["content"])

    result_df = run_matching(
        vendor_df,
        employee_df,
        v_col_dropdown.value,
        e_col_dropdown.value,
        threshold_slider.value
    )

    if result_df.empty:
        print("Nothing to export.")
        return

    buffer = BytesIO()
    with pd.ExcelWriter(buffer, engine="openpyxl") as writer:
        result_df.to_excel(writer, index=False, sheet_name="COI_RESULTS")

    buffer.seek(0)
    display(buffer)

export_button.on_click(export_excel)
display(export_button)


Button(button_style='primary', description='RUN ANALYSIS', style=ButtonStyle())

Output()

Button(button_style='success', description='Download Excel Report', style=ButtonStyle())